In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib, os

# === 1. Indlæs data ===
data_path = Path("breast+cancer+wisconsin+diagnostic") / "wdbc.data"

cols = ["id","diagnosis",
"radius_mean","texture_mean","perimeter_mean","area_mean","smoothness_mean","compactness_mean",
"concavity_mean","concave_points_mean","symmetry_mean","fractal_dimension_mean",
"radius_se","texture_se","perimeter_se","area_se","smoothness_se","compactness_se",
"concavity_se","concave_points_se","symmetry_se","fractal_dimension_se",
"radius_worst","texture_worst","perimeter_worst","area_worst","smoothness_worst","compactness_worst",
"concavity_worst","concave_points_worst","symmetry_worst","fractal_dimension_worst"]

df = pd.read_csv(data_path, header=None, names=cols)

# === 2.  ===
df = df.drop(columns=["id"])
df["diagnosis"] = df["diagnosis"].map({"M":1, "B":0}).astype(int)

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

# === 3. Split og træn model ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

# === 4. Evaluer ===
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

# === 5. Gem model ===
os.makedirs("models", exist_ok=True)
joblib.dump({"scaler": scaler, "model": clf, "features": X.columns.tolist()}, "models/breast_model.joblib")
print("Model gemt i models/breast_model.joblib")


Accuracy: 0.9649122807017544
F1-score: 0.9512195121951219
ROC AUC: 0.996031746031746
Model gemt i models/breast_model.joblib


In [7]:
# Gem bundle igen - denne gang med metrics og feature_stats
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "f1": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
}

feature_stats = {
    "min": X.min().to_dict(),
    "max": X.max().to_dict(),
    "median": X.median().to_dict(),
}

joblib.dump(
    {
        "scaler": scaler,
        "model": clf,
        "features": X.columns.tolist(),
        "metrics": metrics,
        "feature_stats": feature_stats,
    },
    "models/breast_model.joblib",
)

print("Bundle gemt med metrics + feature_stats → models/breast_model.joblib")
metrics


Bundle gemt med metrics + feature_stats → models/breast_model.joblib


{'accuracy': 0.9649122807017544,
 'f1': 0.9512195121951219,
 'roc_auc': 0.996031746031746}

In [9]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib, os

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "f1": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
}

feature_stats = {
    "min": X.min().to_dict(),
    "max": X.max().to_dict(),
    "median": X.median().to_dict(),
}

os.makedirs("models", exist_ok=True)
joblib.dump(
    {
        "scaler": scaler,
        "model": clf,
        "features": X.columns.tolist(),
        "metrics": metrics,
        "feature_stats": feature_stats,
    },
    "models/breast_model.joblib",
)
print("Bundle gemt med metrics + feature_stats")
metrics


Bundle gemt med metrics + feature_stats


{'accuracy': 0.9649122807017544,
 'f1': 0.9512195121951219,
 'roc_auc': 0.996031746031746}

## Udvidet eksperiment: flere modeller og cross-validation


In [18]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib, os

X_arr, y_arr = X.values, y.values

# 1) CV-score for baseline (LogReg)
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000))
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc_log = cross_val_score(logreg, X_arr, y_arr, cv=cv, scoring="roc_auc")
cv_acc_log = cross_val_score(logreg, X_arr, y_arr, cv=cv, scoring="accuracy")

# 2) GridSearch på RandomForest
rf = RandomForestClassifier(random_state=42)
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 8, 16],
    "min_samples_split": [2, 5],
}
gs = GridSearchCV(rf, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
gs.fit(X_arr, y_arr)
best_rf = gs.best_estimator_

# 3) Hold-out eval for begge (samme split som før)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)
# LogReg med scaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
logreg_clf = LogisticRegression(max_iter=5000).fit(X_train_s, y_train)

# RF på rå features
best_rf.fit(X_train, y_train)

# Metrics
def eval_model(clf, X_trf, X_tef, scaled=False):
    if scaled:
        y_proba = clf.predict_proba(X_tef)[:,1]
    else:
        y_proba = clf.predict_proba(X_tef)[:,1]
    y_pred = (y_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "f1": float(f1_score(y_test, y_pred)),
        "roc_auc": float(roc_auc_score(y_test, y_proba)),
        "fpr_tpr": roc_curve(y_test, y_proba)[:2]  # (fpr, tpr)
    }

m_log = eval_model(logreg_clf, X_train_s, X_test_s, scaled=True)
m_rf  = eval_model(best_rf, X_train, X_test, scaled=False)

# 4) Vælg bedste model på ROC AUC
best_name, best_bundle = max(
    [("logreg", m_log["roc_auc"]), ("random_forest", m_rf["roc_auc"])],
    key=lambda t: t[1]
)[0], None

if best_name == "logreg":
    best_bundle = {
        "type": "logreg",
        "scaler": scaler,
        "model": logreg_clf
    }
else:
    best_bundle = {
        "type": "random_forest",
        "scaler": None,
        "model": best_rf
    }

# 5) Gem alt vi skal bruge i app’en
feature_stats = {"min": X.min().to_dict(), "max": X.max().to_dict(), "median": X.median().to_dict()}
bundle = {
    "best_model_name": best_name,
    "best_model_auc": float(m_log["roc_auc"] if best_name=="logreg" else m_rf["roc_auc"]),
    "models": {
        "logreg": {
            "metrics": {k:v for k,v in m_log.items() if k!="fpr_tpr"},
            "fpr": m_log["fpr_tpr"][0].tolist(),
            "tpr": m_log["fpr_tpr"][1].tolist()
        },
        "random_forest": {
            "metrics": {k:v for k,v in m_rf.items() if k!="fpr_tpr"},
            "fpr": m_rf["fpr_tpr"][0].tolist(),
            "tpr": m_rf["fpr_tpr"][1].tolist()
        }
    },
    "features": X.columns.tolist(),
    "feature_stats": feature_stats,
    # lagre scaler separat hvis logreg er valgt
    "scaler_for_logreg": scaler
}
os.makedirs("models", exist_ok=True)
joblib.dump(bundle, "models/breast_model.joblib")
print("Saved bundle with models + CV + ROC data")


Saved bundle with models + CV + ROC data
